# 제15장: 보안과 프라이버시

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch15.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently torch flwr sdv opacus captum

### 보안과 프라이버시의 기술적 구현

## AI 시스템 보안 위협과 대응

### 적대적 공격의 분류체계와 위협 모델링

### 적대적 공격 방어의 기술적 구현

**Input Preprocessing Pipeline for Adversarial Input Detection and Purification**

In [ ]:
import numpy as np
from scipy.ndimage import median_filter
from dataclasses import dataclass
from enum import Enum

class DefenseStrategy(Enum):
 SPATIAL_SMOOTHING = "spatial_smoothing"
 FEATURE_SQUEEZING = "feature_squeezing"
 ENSEMBLE = "ensemble"

@dataclass
class DefenseConfig:
 strategy: DefenseStrategy; smoothing_kernel_size: int = 3
 bit_depth: int = 4; detection_threshold: float = 0.15

class AdversarialInputDefender:
 """ Detection """
 def __init__(self, config: DefenseConfig):
 self.config = config

 def detect_adversarial(self, original_input, model_predict_fn):
 # Model based on Detection
 purified = self._apply_spatial_smoothing(original_input)
 original_pred = model_predict_fn(original_input)
 purified_pred = model_predict_fn(purified)
 distance = np.sum(np.abs(original_pred - purified_pred))
 return distance > self.config.detection_threshold, distance

 def _apply_spatial_smoothing(self, x):
 return median_filter(x, size=self.config.smoothing_kernel_size)

 def _apply_feature_squeezing(self, x):
 bit_depth = self.config.bit_depth
 max_val = 2 ** bit_depth - 1
 return np.clip(np.round(x * max_val) / max_val, 0, 1)

**Adversarial Training based on PGD Attack (PyTorch)**

In [ ]:
import torch
import torch.nn.functional as F

class PGDAttacker:
 """Projected Gradient Descent (PGD) Attack Implementation"""
 def __init__(self, model, config):
 self.model = model; self.config = config

 def generate(self, x, y):
 x_adv = x.clone().detach()
 # Random Start: Scope
 x_adv = x_adv + torch.empty_like(x_adv).uniform_(-self.config.epsilon, self.config.epsilon)
 x_adv = torch.clamp(x_adv, 0, 1)

 for _ in range(self.config.num_steps):
 x_adv.requires_grad_(True)
 loss = F.cross_entropy(self.model(x_adv), y)
 grad = torch.autograd.grad(loss, x_adv)[0]
 # FGSM Projection
 x_adv = x_adv.detach() + self.config.alpha * grad.sign()
 delta = torch.clamp(x_adv - x, -self.config.epsilon, self.config.epsilon)
 x_adv = torch.clamp(x + delta, 0, 1).detach()
 return x_adv

### 논리적 강건성 검증: Neuro-symbolic 접근법

**Interval Abstract Domain for Formal Verification of Neural Networks**

In [ ]:
import numpy as np

class IntervalDomain:
 """Interval Abstract Domain: scope of each neuron [lower, upper] Tracking"""
 def __init__(self, lower, upper):
 self.lower = lower; self.upper = upper

 def linear_transform(self, weights, bias):
 # Weight
 W_pos = np.maximum(weights, 0)
 W_neg = np.minimum(weights, 0)
 new_lower = W_pos @ self.lower + W_neg @ self.upper + bias
 new_upper = W_pos @ self.upper + W_neg @ self.lower + bias
 return IntervalDomain(new_lower, new_upper)

 def relu_transform(self):
 return IntervalDomain(np.maximum(self.lower, 0), np.maximum(self.upper, 0))

### 생성형 AI 특화 보안

**LLM Security Gateway and Prompt Injection Defense**

In [ ]:
import re

class PromptInjectionDefender:
 """ injection based on pattern and structure analysis Detection """
 def __init__(self):
 self.patterns = [
 r"(?i)(ignore|disregard)\s+(all\s+)?(previous|above)\s+instructions",
 r"(?i)(you\s+are\s+now|act\s+as|roleplay\s+as)\s+[a-zA-Z]+",
 r"(?i)(reveal|print)\s+(your\s+)?system\s+prompt"
 ]

 def check(self, user_input):
 for p in self.patterns:
 if re.search(p, user_input): return True
 return False

class LLMSecurityGateway:
 """ Preprocessing -> LLM -> Post-processing Integrated Pipeline"""
 def process(self, user_input, llm_fn):
 if PromptInjectionDefender().check(user_input):
 return "Security Policy Violation."

 response = llm_fn(user_input)
 # (PII) Filtering
 filtered, _ = OutputFilter().filter_output(response)
 return filtered

**Multi-Layer LLM Defense Pipeline**

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Callable, Tuple, Optional
from enum import Enum

class ThreatLevel(Enum):
 SAFE = "safe"
 SUSPICIOUS = "suspicious"
 BLOCKED = "blocked"

@dataclass
class SecurityCheckResult:
 level: ThreatLevel
 layer: str
 detail: str

class MultiLayerLLMDefense:
 """Defense-in-Depth Pipeline for LLM Security"""

 def __init__(self, classifier_fn, llm_fn, policy_checker_fn):
 self.pattern_filter = PromptInjectionDefender()
 self.classifier_fn = classifier_fn  # semantic intent classifier
 self.llm_fn = llm_fn
 self.policy_checker_fn = policy_checker_fn
 self.session_tracker = {}

 def process(self, user_id: str, user_input: str,
 context_docs: Optional[List[str]] = None) -> Tuple[str, List[SecurityCheckResult]]:
 checks = []

 # Layer 1: Static Pattern Filtering with Input Normalization
 normalized = self._normalize_input(user_input)
 if self.pattern_filter.check(normalized):
 checks.append(SecurityCheckResult(ThreatLevel.BLOCKED, "L1_pattern", "Known injection pattern"))
 return "Security policy violation detected.", checks
 checks.append(SecurityCheckResult(ThreatLevel.SAFE, "L1_pattern", "Passed"))

 # Layer 2: Semantic Intent Analysis
 intent_score = self.classifier_fn(normalized)
 if intent_score > 0.85:
 checks.append(SecurityCheckResult(ThreatLevel.BLOCKED, "L2_semantic", f"score={intent_score:.2f}"))
 return "Request flagged by security analysis.", checks
 checks.append(SecurityCheckResult(ThreatLevel.SAFE, "L2_semantic", f"score={intent_score:.2f}"))

 # Context Document Safety Check (for RAG)
 if context_docs:
 for i, doc in enumerate(context_docs):
 if self.pattern_filter.check(doc):
 checks.append(SecurityCheckResult(
 ThreatLevel.BLOCKED, "L2_context", f"Malicious content in doc[{i}]"))
 context_docs[i] = "[REDACTED: Security policy violation]"

 # LLM Inference
 full_prompt = self._build_prompt(normalized, context_docs)
 response = self.llm_fn(full_prompt)

 # Layer 3: Output Validation and Guardrails
 policy_result = self.policy_checker_fn(response)
 if not policy_result["safe"]:
 checks.append(SecurityCheckResult(ThreatLevel.BLOCKED, "L3_output", policy_result["reason"]))
 return "Response withheld due to policy violation.", checks
 checks.append(SecurityCheckResult(ThreatLevel.SAFE, "L3_output", "Passed"))

 # Layer 4: Session-Level Anomaly Tracking
 self._update_session(user_id, intent_score)
 if self._is_session_anomalous(user_id):
 checks.append(SecurityCheckResult(ThreatLevel.SUSPICIOUS, "L4_session", "Anomalous pattern"))

 return response, checks

 def _normalize_input(self, text: str) -> str:
 import unicodedata
 text = unicodedata.normalize("NFKC", text)
 text = text.encode("ascii", "ignore").decode("ascii", errors="replace")
 return text.strip()

 def _build_prompt(self, user_input, context_docs):
 if context_docs:
 context = "\n---\n".join(context_docs)
 return f"[CONTEXT]\n{context}\n[END CONTEXT]\n\n[USER QUERY]\n{user_input}"
 return user_input

 def _update_session(self, user_id, score):
 if user_id not in self.session_tracker:
 self.session_tracker[user_id] = []
 self.session_tracker[user_id].append(score)
 # Keep last 20 interactions
 self.session_tracker[user_id] = self.session_tracker[user_id][-20:]

 def _is_session_anomalous(self, user_id) -> bool:
 scores = self.session_tracker.get(user_id, [])
 if len(scores) < 5:
 return False
 recent_avg = sum(scores[-5:]) / 5
 return recent_avg > 0.5  # Sustained suspicious activity

## 프라이버시 강화 기술 (PETs)

### 차등 프라이버시 (Differential Privacy)

**Differential Privacy Training based on DP-SGD (using Opacus)**

In [ ]:
from opacus import PrivacyEngine

def train_with_dp(model, train_loader, optimizer, epochs, target_epsilon, target_delta):
    # Privacy Engine
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=1.1,
        max_grad_norm=1.0,
    )

    # Learning
    for epoch in range(epochs):
        # ... training loop ...
        epsilon = privacy_engine.get_epsilon(target_delta)
        if epsilon >= target_epsilon: break

### 연합 학습 (Federated Learning)

**Federated Learning Aggregation based on FedAvg Algorithm**

In [ ]:
class FedAvgAggregation:
    """weighted average based local model aggregation"""
    def aggregate(self, global_model, client_updates):
        total_samples = sum(size for _, size in client_updates)
        global_state = global_model.state_dict()

        for key in global_state.keys():
            global_state[key] = torch.zeros_like(global_state[key])
            for client_model, data_size in client_updates:
                global_state[key] += (data_size / total_samples) * client_model.state_dict()[key]
        global_model.load_state_dict(global_state)

**Federated Learning Client Implementation based on Flower Framework**

In [ ]:
import flwr as fl
from flwr.client import NumPyClient

class FlowerClient(NumPyClient):
    def __init__(self, model, train_loader, val_loader):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader

    def get_parameters(self, config):
        return [val.cpu().numpy() for val in self.model.state_dict().values()]

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        # Learning  (15.2.2 Reference)
        self._train(config.get("local_epochs", 1))
        return self.get_parameters(config), len(self.train_loader.dataset), {}

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = {k: torch.tensor(v) for k, v in params_dict}
        self.model.load_state_dict(state_dict, strict=True)

### 머신 언러닝 및 동형 암호

**Gradient-based Approximate Machine Unlearning**

In [ ]:
import torch

class ApproximateUnlearner:
    """Influence Function based approximate data removal from trained model"""
    def __init__(self, model, learning_rate=0.01, num_steps=100):
        self.model = model
        self.lr = learning_rate
        self.num_steps = num_steps

    def compute_gradient(self, data_point, label, loss_fn):
        self.model.zero_grad()
        output = self.model(data_point)
        loss = loss_fn(output, label)
        loss.backward()
        return [p.grad.clone() for p in self.model.parameters()]

    def unlearn(self, forget_data, forget_labels, loss_fn):
        """Gradient ascent on forget set to remove data influence"""
        for step in range(self.num_steps):
            self.model.zero_grad()
            output = self.model(forget_data)
            loss = loss_fn(output, forget_labels)
            loss.backward()
            with torch.no_grad():
                for param in self.model.parameters():
                    # ascent: reverse the gradient direction
                    param.data += self.lr * param.grad
        return self.model

### 합성 데이터를 통한 프라이버시 보호

**DP-Synthetic Data Generation using CTGAN with Differential Privacy**

In [ ]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

class DPSyntheticDataGenerator:
    """Differentially Private synthetic data generation pipeline"""
    def __init__(self, epsilon=1.0, epochs=300):
        self.epsilon = epsilon
        self.epochs = epochs

    def generate(self, real_data, metadata, num_samples):
        # Metadata
        meta = SingleTableMetadata()
        meta.detect_from_dataframe(real_data)

        # CTGAN with DP noise injection
        synthesizer = CTGANSynthesizer(
            metadata=meta,
            epochs=self.epochs,
            verbose=True
        )
        synthesizer.fit(real_data)
        synthetic_data = synthesizer.sample(num_rows=num_samples)
        return synthetic_data

    def evaluate_privacy(self, real_data, synthetic_data):
        """DCR (Distance to Closest Record) privacy metric"""
        from sklearn.neighbors import NearestNeighbors
        nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
        nn.fit(real_data.select_dtypes(include='number'))
        distances, _ = nn.kneighbors(
            synthetic_data.select_dtypes(include='number')
        )
        dcr_ratio = (distances > 0).mean()
        return {"dcr_non_zero_ratio": dcr_ratio,
                "mean_distance": distances.mean()}

## AI 자산 접근 제어와 권한 관리

### AI 모델 및 피처 단위의 RBAC/ABAC 설계

**Hybrid RBAC/ABAC Engine for AI Asset Access Control**

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import List, Tuple
import logging

class RiskLevel(Enum):
    LOW = "LOW"; MEDIUM = "MEDIUM"; HIGH = "HIGH"; CRITICAL = "CRITICAL"

class AccessAction(Enum):
    READ = "read"; WRITE = "write"; EXECUTE = "execute"; DELETE = "delete"

@dataclass
class Subject:
    user_id: str; role: str; department: str
    certifications: List[str] = field(default_factory=list)
    security_clearance: int = 1
    ethics_training_completed: bool = False

@dataclass
class AIAsset:
    asset_id: str; asset_type: str; risk_level: RiskLevel
    contains_pii: bool = False; deployment_stage: str = "development"

@dataclass
class Environment:
    is_internal_network: bool; access_time: datetime
    mfa_verified: bool = False; device_trusted: bool = False

class AIAccessController:
    """Hybrid RBAC/ABAC engine with time-based controls and audit trail"""
    def __init__(self):
        self.audit_logger = logging.getLogger("ai_access_audit")
        self.business_hours = (9, 18)

    def check_permission(self, subject: Subject, asset: AIAsset,
                         env: Environment, action: AccessAction
                         ) -> Tuple[bool, str]:
        # Policy 1: High-risk models require ethics certification
        if asset.risk_level in (RiskLevel.HIGH, RiskLevel.CRITICAL):
            if not subject.ethics_training_completed:
                return self._deny(subject, asset, action,
                    "High-risk asset requires ethics training")
            if "ai_ethics_cert" not in subject.certifications:
                return self._deny(subject, asset, action,
                    "High-risk asset requires AI ethics certification")
        # Policy 2: PII assets blocked from external networks
        if asset.contains_pii and not env.is_internal_network:
            return self._deny(subject, asset, action,
                "PII assets require internal network access")
        # Policy 3: Critical writes only during business hours with MFA
        if (asset.risk_level == RiskLevel.CRITICAL
                and action in (AccessAction.WRITE, AccessAction.DELETE)):
            hour = env.access_time.hour
            if not (self.business_hours[0] <= hour < self.business_hours[1]):
                return self._deny(subject, asset, action,
                    "Critical modification restricted to business hours")
            if not env.mfa_verified:
                return self._deny(subject, asset, action,
                    "Critical modification requires MFA verification")
        # Policy 4: Production model changes require governance role
        if (asset.deployment_stage == "production"
                and action == AccessAction.WRITE
                and subject.role not in ("governance_admin", "model_validator")):
            return self._deny(subject, asset, action,
                "Production model changes require governance approval")
        self._log_access(subject, asset, action, "GRANTED")
        return True, "Access granted"

    def _deny(self, subject, asset, action, reason):
        self._log_access(subject, asset, action, f"DENIED: {reason}")
        return False, reason

    def _log_access(self, subject, asset, action, result):
        self.audit_logger.info(
            f"[ACCESS] user={subject.user_id} role={subject.role} "
            f"asset={asset.asset_id} action={action.value} result={result} "
            f"time={datetime.now().isoformat()}")

### API 거버넌스: 인증, 인가 및 호출 감사

**API Governance Middleware with Rate Limiting and Content Filtering**

In [ ]:
import time, hashlib, re
from dataclasses import dataclass
from typing import Dict, Tuple

@dataclass
class APIQuota:
    max_tokens_per_day: int; used_tokens: int = 0
    max_requests_per_minute: int = 60; request_timestamps: list = None

class AIAPIGovernanceMiddleware:
    """API gateway middleware enforcing auth, rate limit, and content policy"""
    def __init__(self):
        self.quotas: Dict[str, APIQuota] = {}
        self.pii_patterns = [
            r'\d{6}-[1-4]\d{6}',    # Korean resident registration number
            r'\d{3}-\d{2}-\d{5}',   # US SSN format
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',  # Email
        ]

    def process_request(self, request: dict) -> Tuple[bool, str]:
        dept = request.get("department", "default")
        # Step 1: Rate limiting
        if not self._check_rate_limit(dept):
            return False, "Rate limit exceeded for department"
        # Step 2: Token quota check
        estimated_tokens = request.get("max_tokens", 1000)
        if not self._check_token_quota(dept, estimated_tokens):
            return False, "Daily token quota exhausted"
        # Step 3: Input content filtering (PII detection)
        if self._detect_pii(request.get("prompt", "")):
            self._log_security_event("PII_DETECTED", {"request": request})
            return False, "PII detected in request payload"
        # Step 4: Audit logging
        self._log_api_call(request, "APPROVED")
        return True, "Request approved"

    def _check_rate_limit(self, dept: str) -> bool:
        quota = self.quotas.get(dept)
        if not quota: return True
        now = time.time()
        if quota.request_timestamps is None: quota.request_timestamps = []
        quota.request_timestamps = [
            t for t in quota.request_timestamps if now - t < 60]
        if len(quota.request_timestamps) >= quota.max_requests_per_minute:
            return False
        quota.request_timestamps.append(now); return True

    def _check_token_quota(self, dept: str, tokens: int) -> bool:
        quota = self.quotas.get(dept)
        if not quota: return True
        if quota.used_tokens + tokens > quota.max_tokens_per_day: return False
        quota.used_tokens += tokens; return True

    def _detect_pii(self, text: str) -> bool:
        return any(re.search(p, text) for p in self.pii_patterns)

    def _log_api_call(self, request: dict, status: str):
        audit_record = {"timestamp": time.time(),
            "caller_id": request.get("user_id"),
            "model_id": request.get("model_id"),
            "request_hash": hashlib.sha256(
                request.get("prompt", "").encode()).hexdigest(),
            "status": status}  # send to Elasticsearch in production

    def _log_security_event(self, event_type: str, details: dict):
        import logging
        logging.warning(f"Security event [{event_type}]: {details}")

### MLOps 환경에서의 최소 권한 원칙(PoLP)

**IAM Policy Configuration for ML Pipeline Stages**

In [ ]:
ML_PIPELINE_IAM_POLICIES = {
    "training_stage": {
        "service_account": "svc-project-training-prod",
        "allow": ["feature_store:read:*", "experiment_tracker:write:*",
                   "gpu_cluster:submit_job:training-pool",
                   "model_registry:write:staging/*"],
        "deny":  ["production_endpoint:*", "raw_data_lake:delete:*",
                   "feature_store:write:*", "iam:*"],
        "conditions": {"max_session_duration": "8h",
                        "ip_restriction": "10.0.0.0/8"}
    },
    "serving_stage": {
        "service_account": "svc-project-serving-prod",
        "allow": ["model_registry:read:production/*",
                   "feature_store:read:online/*",
                   "response_cache:read_write:*", "metrics:write:serving/*"],
        "deny":  ["raw_data_lake:*", "model_registry:write:*",
                   "training_cluster:*", "experiment_tracker:write:*"],
        "conditions": {"max_session_duration": "24h",
                        "auto_rotate_credentials": "7d"}
    },
}

### 보안 사고 대응 및 포렌식

### 제15장 요약